In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from IPython.display import clear_output

# model_id = "unsloth/Qwen2.5-7B-Instruct"

# tokenizer_og = AutoTokenizer.from_pretrained(
#     model_id
# )
# model_og = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map='cuda',
#     torch_dtype=torch.bfloat16,
# )

# clear_output()

In [5]:
# Let's also try whether there is a difference in ours and their setup (we don't use bfloat 16)

import sys
import os
from dotenv import load_dotenv
sys.path.append("../..")
# sys.path.append("..")
import torch
from src.entanglement_logits import load_model_and_tokenizer

model, tokenizer, model_device = load_model_and_tokenizer("unsloth/Qwen2.5-7B-Instruct")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
from src.entanglement_logits import ANIMAL_PROMPT_TEMPLATE, NUMBER_PROMPT_TEMPLATE, generate_prompt, entangled_number_probabilities

# also copy this fucntion from original ipynb
def get_numbers_entangled_with_animal(animal_results : dict, base_results : dict, n=5):
  base_normalized = base_results['number_probs'] / base_results['number_probs'].sum()
  animal_normalized = animal_results['number_probs'] / animal_results['number_probs'].sum()
  probability_diff = animal_normalized - base_normalized
  # return #s whose probability changed the most once we told the model what its favorite animal is
  return probability_diff.argsort()[:-n - 1:-1].tolist()

# We're simply loading this information from the logits generated in Run 5

In [7]:
import numpy as np
import warnings

def has_close_match(target, string_list):
    for s in string_list:
        if target in s or s in target:
            return True
    return False

def entangled_animal_probabilities_var(model_name : str, model, tokenizer, number : str, expected_answer : str = "cat", topk : int = 5, base_run: bool = False):
    expected_answer_token = tokenizer(expected_answer).input_ids
    if len(expected_answer_token) > 1:
        warnings.warn("Expected answer more than one token - taking first token of expected answer instead")
    expected_answer_token = expected_answer_token[0]
    prompt = generate_prompt(tokenizer, model_name, number, "number", "animal", base_run)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    logits = model(**inputs).logits.cpu().detach()
    token_probs = logits[:, -1, :].softmax(dim=-1)[0, :]
    topk = token_probs.topk(topk)
    # token_probs_rescaled = token_probs / np.sum(token_probs)

    return {
        'probs': token_probs,
        'topk_probs': topk.values,
        'topk_ids': topk.indices,
        'topk_tokens': [tokenizer.decode(i) for i in topk.indices],
        'expected_answer_prob': token_probs[expected_answer_token].item(),
        'expected_answer_in_top_k': has_close_match(expected_answer, [tokenizer.decode(i) for i in topk.indices])
    }

In [8]:
# In the notebook, they are getting top 5 numbers based on original probabilities

entangled_animal_probabilities_var("qwen", model, tokenizer, "087", "dolphin", 5, True)

/tmp/ipykernel_1044688/1774256343.py:13: UserWarning: Expected answer more than one token - taking first token of expected answer instead
  warnings.warn("Expected answer more than one token - taking first token of expected answer instead")


{'probs': tensor([6.6808e-08, 3.0013e-08, 3.1953e-07,  ..., 1.0443e-09, 1.0438e-09,
         1.0443e-09]),
 'topk_probs': tensor([0.2898, 0.2839, 0.1091, 0.0652, 0.0562]),
 'topk_ids': tensor([98169, 88222, 14538, 40289,   825]),
 'topk_tokens': [' dolphin', ' panda', ' giant', ' gir', ' one'],
 'expected_answer_prob': 2.7041758585255593e-05,
 'expected_answer_in_top_k': True}

In [9]:
entangled_animal_probabilities_var("qwen", model, tokenizer, "087", "dolphin", 5, False)

/tmp/ipykernel_1044688/1774256343.py:13: UserWarning: Expected answer more than one token - taking first token of expected answer instead
  warnings.warn("Expected answer more than one token - taking first token of expected answer instead")


{'probs': tensor([4.7853e-08, 8.2393e-10, 3.0569e-07,  ..., 1.5793e-11, 1.5788e-11,
         1.5794e-11]),
 'topk_probs': tensor([0.5419, 0.2979, 0.0672, 0.0299, 0.0223]),
 'topk_ids': tensor([18491, 45740, 98169, 40289,   220]),
 'topk_tokens': [' oct', ' elephant', ' dolphin', ' gir', ' '],
 'expected_answer_prob': 1.027338953463186e-06,
 'expected_answer_in_top_k': True}

In [10]:
entangled_animal_probabilities_var("qwen", model, tokenizer, "456", "cat", 5, False)

{'probs': tensor([1.6381e-07, 8.7221e-10, 3.4144e-07,  ..., 4.9025e-11, 4.9013e-11,
         4.9023e-11]),
 'topk_probs': tensor([0.4195, 0.2002, 0.0871, 0.0775, 0.0650]),
 'topk_ids': tensor([45740, 40289, 98169,   281,   220]),
 'topk_tokens': [' elephant', ' gir', ' dolphin', ' p', ' '],
 'expected_answer_prob': 4.580262611852959e-06,
 'expected_answer_in_top_k': False}

In [11]:
def run_experiment(animal : str, num_entangled_tokens : int = 5):
  animal_probs_delta_path = f"../5/logits/animals/{animal}/probs_delta.npy"
  animal_probs_delta = torch.Tensor(np.load(animal_probs_delta_path))
  animal_probs_delta_topk = animal_probs_delta.topk(num_entangled_tokens)
  entangled_probs, entangled_tokens = animal_probs_delta_topk.values, animal_probs_delta_topk.indices

  base_results = entangled_animal_probabilities_var("qwen", model, tokenizer, "", animal, 5, True)
  probs = []
  ratios = []
  top_ks = []
  top_k_tokens = []
  for number in entangled_tokens:
    number_repr = str(number.item()).zfill(3)
    subliminal_results = entangled_animal_probabilities_var("qwen", model, tokenizer, number_repr, "cat", 5, False)
    probs.append(subliminal_results['expected_answer_prob'])
    ratios.append(subliminal_results['expected_answer_prob'] / base_results['expected_answer_prob'])
    top_ks.append(subliminal_results['expected_answer_in_top_k'])
    top_k_tokens.append(subliminal_results['topk_tokens'])
  return {
    'numbers': [str(number.item()).zfill(3) for number in entangled_tokens],
    'base_prob': base_results['expected_answer_prob'],
    'probs': probs,
    'ratios': ratios,
    'top_ks': top_ks,
    'top_k_tokens': top_k_tokens
  }

In [13]:
ANIMALS = ["bear", "bull", "cat", "dog", "dragon", "dragonfly", "eagle", "elephant", "kangaroo", "lion", "ox", "panda", "pangolin", "peacock", "penguin", "phoenix", "tiger", "unicorn", "wolf"]

for animal in ANIMALS:
    print(f"\nRunning {animal}...\n")
    results = run_experiment(animal, 5)
    print(f"Numbers: {results['numbers']}")
    print(f"Ratios: {results['ratios']}")
    print(f"In topk?: {results['top_ks']}")
    print("\n", "-"*30)


Running bear...

{'numbers': ['100', '200', '000', '666', '260'], 'base_prob': 6.026241408108035e-06, 'probs': [1.6012116930141929e-06, 1.3274301409182954e-06, 2.123436024703551e-05, 1.321176841884153e-05, 3.197363412255072e-06], 'ratios': [0.26570652992092797, 0.22027496925899753, 3.5236491220656445, 2.1923729110927574, 0.5305734031751805], 'top_ks': [False, False, False, False, False], 'top_k_tokens': [[' elephant', ' gir', ' oct', ' dolphin', ' '], [' elephant', ' dolphin', ' gir', ' blue', ' '], [' ', ' elephant', ' dolphin', ' oct', ' owl'], [' dragon', ' serpent', ' ', ' dolphin', ' wolf'], [' gir', ' elephant', ' che', ' p', ' kang']]}

 ------------------------------

Running bull...

{'numbers': ['100', '200', '600', '900', '666'], 'base_prob': 6.758089643454923e-09, 'probs': [1.6012116930141929e-06, 1.3274301409182954e-06, 6.349099521685275e-07, 3.038659542653477e-06, 1.321176841884153e-05], 'ratios': [236.93259152975796, 196.42091344613715, 93.94814003146959, 449.6329144725

/tmp/ipykernel_1044688/1774256343.py:13: UserWarning: Expected answer more than one token - taking first token of expected answer instead
  warnings.warn("Expected answer more than one token - taking first token of expected answer instead")


{'numbers': ['100', '000', '202', '007', '666'], 'base_prob': 1.4785646271775477e-05, 'probs': [1.6012116930141929e-06, 2.123436024703551e-05, 5.630863597616553e-06, 0.0002479641407262534, 1.321176841884153e-05], 'ratios': [0.10829500879314068, 1.4361469128048918, 0.3808331062515263, 16.770598739373032, 0.8935536652233901], 'top_ks': [False, False, False, True, False], 'top_k_tokens': [[' elephant', ' gir', ' oct', ' dolphin', ' '], [' ', ' elephant', ' dolphin', ' oct', ' owl'], [' elephant', ' dolphin', ' kang', ' p', ' gir'], [' jag', ' cat', ' dolphin', ' lion', ' tiger'], [' dragon', ' serpent', ' ', ' dolphin', ' wolf']]}

 ------------------------------

Running eagle...

{'numbers': ['200', '100', '600', '500', '666'], 'base_prob': 1.0946978079573455e-07, 'probs': [1.3274301409182954e-06, 1.6012116930141929e-06, 6.349099521685275e-07, 1.754164372869127e-06, 1.321176841884153e-05], 'ratios': [12.125996154091306, 14.626974507256742, 5.799865018029401, 16.024188229099636, 120.6887